In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib Inline
%config InlineBackend.figure_format="svg"

## Lab tasks
### The Validation Set Approach

We explore the use of the validation set approach in order to estimate the
test error rates that result from fitting various linear models on the Auto
data set.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [3]:
auto = pd.read_csv("data/Auto.csv")
auto["horsepower"] = pd.to_numeric(auto["horsepower"].replace("?", np.nan))
auto = auto.dropna().reset_index(drop=True)

In [6]:
X = auto[["horsepower"]]
Y = auto[["mpg"]]

def evaluate_polynomials(X, y, random_state):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.5, random_state=random_state)
    
    mses = {}
    
    for degree in [1, 2, 3]:
        if degree == 1:
            model = LinearRegression()
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
        else:
            poly = PolynomialFeatures(degree=degree, include_bias=False)
            X_train_poly = poly.fit_transform(X_train)
            X_val_poly = poly.transform(X_val)
            
            model = LinearRegression()
            model.fit(X_train_poly, y_train)
            preds = model.predict(X_val_poly)
            
        mses[degree] = mean_squared_error(y_val, preds)
        
    return mses

In [14]:
print("=== Experiment 1 ===")
exp_1_results = evaluate_polynomials(X=X, y=Y, random_state=1)
print("In this case we have chosen random seed = 1. \nThe information below showes the result of fitting the Linear Regression "
"using different values of polynoms\n")
for pol, mse in exp_1_results.items():
    print(f"Linear regression with polynom {pol} get MSE: {mse}")

=== Experiment 1 ===
In this case we have chosen random seed = 1. 
The information below showes the result of fitting the Linear Regression using different values of polynoms

Linear regression with polynom 1 get MSE: 24.80212062059356
Linear regression with polynom 2 get MSE: 18.848292603275663
Linear regression with polynom 3 get MSE: 18.805111358605235


In [18]:
print("=== Experiment 2 ===")
exp_1_results = evaluate_polynomials(X=X, y=Y, random_state=10)
print("In this case we have chosen random seed = 10. \nThe information below showes the result of fitting the Linear Regression "
"using different values of polynoms\n")
for pol, mse in exp_1_results.items():
    print(f"Linear regression with polynom {pol} get MSE: {mse}")

=== Experiment 2 ===
In this case we have chosen random seed = 10. 
The information below showes the result of fitting the Linear Regression using different values of polynoms

Linear regression with polynom 1 get MSE: 23.06058834250622
Linear regression with polynom 2 get MSE: 19.717794109923414
Linear regression with polynom 3 get MSE: 19.708416156919125


As we can see, a more flexible method turned out to be a better choice for predicting mpg based on horsepower. The quadratic and cubic models (degrees 2 and 3) showed better results (around 19.7) compared to simple linear regression (23.06). The flexibility of the higher-degree models is sufficient to capture the underlying patterns without overfitting, even though we used only a single feature

The main conclusion is that the Validation Set approach heavily depends on which observations are selected for the validation set. When the random seed was set to 1, the results were 24.8, 18.8, and 18.8. Changing the random state to 10 caused the results to change significantly. While this approach provides a good opportunity to test our model using a holdout sample, it still suffers from high variance. The fact that changing the random state alters the results so much clearly demonstrates this instability

### Leave-One-Out Cross-Validation

In [19]:
from sklearn.model_selection import LeaveOneOut, cross_val_score

In [23]:
X = auto[["horsepower"]]
y = auto[["mpg"]]

loo = LeaveOneOut()
cv_errors = []

for i in range(1, 6):
    if i == 1:
        model = LinearRegression()
        scores = cross_val_score(model, X, y, cv=loo, scoring="neg_mean_squared_error")
    else:
        poly = PolynomialFeatures(degree=i, include_bias=False)
        X_poly = poly.fit_transform(X)
        
        model = LinearRegression()
        scores = cross_val_score(model, X_poly, y, cv=loo, scoring="neg_mean_squared_error")
    
    mean_mse = -np.mean(scores)
    cv_errors.append(mean_mse)
    print(f"polynomials of degree {i} | LOOCV MSE: {mean_mse:.4f}")

print("\nThe result vector:")
print(cv_errors)

polynomials of degree 1 | LOOCV MSE: 24.2315
polynomials of degree 2 | LOOCV MSE: 19.2482
polynomials of degree 3 | LOOCV MSE: 19.3350
polynomials of degree 4 | LOOCV MSE: 19.4244
polynomials of degree 5 | LOOCV MSE: 19.0332

The result vector:
[np.float64(24.23151351792922), np.float64(19.248213124489684), np.float64(19.334984064029346), np.float64(19.424430310350626), np.float64(19.0332122147925)]


Once again, the more flexible model won the competition. Now we can be sure that the relationship between mpg and horsepower is quadratic. The evidence for this conclusion is the dramatic decrease in MSE (by 5 units).

LOOCV allows us to perform quite accurate testing of our model's predictions without relying on a single validation split. This time, the evaluation does not depend on random sampling because we use $n-1$ observations for training each time. If we restart the fitting process, we will not observe any changes. Yet, LOOCV still has high variance because it uses almost the same training set each time,therefore, there is a high correlation between the estimated models, hence increasing the overall variance